In [1]:
from pathlib import Path

MODE = "offline"
SEED = 2020
EPOCHS = 5
BATCH_SIZE = 256
LR = 1e-3
TOPK = 5
ALPHA = 0.5

PROJECT_ROOT = Path.cwd().resolve()
SAVE_PATH = PROJECT_ROOT / "data" / "processed" / "temp_results"
DATA_PATH = PROJECT_ROOT / "data" / "raw" / "news_recommendation"
offline = MODE == "offline"

print("=" * 70)
print("RankMixer fair ablation notebook")
print(f"MODE: {MODE}")
print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"SAVE_PATH: {SAVE_PATH}")
print(f"DATA_PATH: {DATA_PATH}")
print("=" * 70)


RankMixer fair ablation notebook
MODE: offline
PROJECT_ROOT: /Users/lixiang/Desktop/funrec-new-rec
SAVE_PATH: /Users/lixiang/Desktop/funrec-new-rec/data/processed/temp_results
DATA_PATH: /Users/lixiang/Desktop/funrec-new-rec/data/raw/news_recommendation


In [2]:
from __future__ import annotations

import copy
import random
import time
from collections import OrderedDict, defaultdict
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Dict, List, Optional, Sequence

import numpy as np
import pandas as pd
from sklearn.metrics import ndcg_score, roc_auc_score
from sklearn.preprocessing import MinMaxScaler

try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    from torch.utils.data import DataLoader, Dataset

    TORCH_AVAILABLE = True
    TORCH_IMPORT_ERROR = None
except ImportError as exc:  # pragma: no cover - environment dependent
    torch = nn = F = DataLoader = Dataset = None
    TORCH_AVAILABLE = False
    TORCH_IMPORT_ERROR = exc

TorchModuleBase = nn.Module if TORCH_AVAILABLE else object
TorchDatasetBase = Dataset if TORCH_AVAILABLE else object


SPARSE_FEATURES = [
    "user_id",
    "click_article_id",
    "category_id",
    "click_environment",
    "click_deviceGroup",
    "click_os",
    "click_country",
    "click_region",
    "click_referrer_type",
]

DENSE_FEATURES = [
    "sim0",
    "time_diff0",
    "word_diff0",
    "sim_max",
    "sim_min",
    "sim_sum",
    "sim_mean",
    "score",
    "rank",
    "click_size",
    "time_diff_mean",
    "active_level",
    "user_time_hob1",
    "user_time_hob2",
    "words_hbo",
    "words_count",
    "is_cat_hab",
    "article_hot_level",
    "article_user_num",
    "article_time_diff_mean",
]

HISTORY_FEATURE = "hist_click_article_id"
DEFAULT_EMB_DIM = 8
DEFAULT_MAX_HIST_LEN = 50
DEFAULT_MODEL_CONFIG = {
    "emb_dim": DEFAULT_EMB_DIM,
    "max_len": DEFAULT_MAX_HIST_LEN,
    "shared_dnn_units": [256, 128],
    "logit_head_units": [64],
    "attention_hidden_units": [64, 32],
    "dropout_rate": 0.1,
}
DEFAULT_RANKMIXER_SEARCH_SPACE = {
    "token_dim": [48, 60, 72, 84, 96, 120],
    "num_layers": [1, 2],
    "expansion_ratio": [2, 4],
    "num_T": 6,
    "num_H": 6,
}

TOKEN_GROUPS = OrderedDict(
    [
        (
            "user_context_sparse",
            {
                "kind": "sparse",
                "features": [
                    "user_id",
                    "click_environment",
                    "click_deviceGroup",
                    "click_os",
                    "click_country",
                    "click_region",
                    "click_referrer_type",
                ],
            },
        ),
        (
            "item_sparse",
            {
                "kind": "sparse",
                "features": ["click_article_id", "category_id"],
            },
        ),
        (
            "recall_similarity_dense",
            {
                "kind": "dense",
                "features": [
                    "sim0",
                    "time_diff0",
                    "word_diff0",
                    "sim_max",
                    "sim_min",
                    "sim_sum",
                    "sim_mean",
                ],
            },
        ),
        (
            "ranking_meta_dense",
            {
                "kind": "dense",
                "features": [
                    "score",
                    "rank",
                    "click_size",
                    "time_diff_mean",
                    "active_level",
                ],
            },
        ),
        (
            "user_article_profile_dense",
            {
                "kind": "dense",
                "features": [
                    "user_time_hob1",
                    "user_time_hob2",
                    "words_hbo",
                    "words_count",
                    "is_cat_hab",
                    "article_hot_level",
                    "article_user_num",
                    "article_time_diff_mean",
                ],
            },
        ),
        ("history_token", {"kind": "history", "features": [HISTORY_FEATURE]}),
    ]
)


def ensure_torch_available():
    if not TORCH_AVAILABLE:  # pragma: no cover - environment dependent
        raise ImportError(
            "rankmixer_fair_ablation requires PyTorch. Install torch and rerun."
        ) from TORCH_IMPORT_ERROR


def get_default_device():
    ensure_torch_available()
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


def clear_torch_cache():
    if TORCH_AVAILABLE and torch.cuda.is_available():
        torch.cuda.empty_cache()


def set_global_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    if TORCH_AVAILABLE:
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)


@dataclass
class LoadedFeatureFrames:
    trn_user_item_feats_df: pd.DataFrame
    val_user_item_feats_df: Optional[pd.DataFrame]
    tst_user_item_feats_df: pd.DataFrame
    click_hist_all: pd.DataFrame
    offline: bool


@dataclass
class MergedFeatureFrames:
    trn_df: pd.DataFrame
    val_df: Optional[pd.DataFrame]
    tst_df: pd.DataFrame
    his_behavior_df: pd.DataFrame
    val_hit_mask: Optional[pd.Series]
    val_df_hit: Optional[pd.DataFrame]
    offline: bool


@dataclass
class PreparedRankingData:
    trn_df: pd.DataFrame
    val_df: Optional[pd.DataFrame]
    tst_df: pd.DataFrame
    val_hit_mask: Optional[pd.Series]
    val_df_hit: Optional[pd.DataFrame]
    sparse_maps: Dict[str, Dict[int, int]]
    sparse_vocab_sizes: Dict[str, int]
    mm_scalers: Dict[str, MinMaxScaler]
    x_trn: Dict[str, np.ndarray]
    y_trn: np.ndarray
    x_val: Optional[Dict[str, np.ndarray]]
    y_val: Optional[np.ndarray]
    x_tst: Optional[Dict[str, np.ndarray]]
    dense_input_dim: int
    emb_dim: int
    max_len: int


def load_feature_frames(save_path: Path, offline: bool = True) -> LoadedFeatureFrames:
    trn_df = pd.read_csv(save_path / "trn_user_item_feats_df_all.csv")
    trn_df["click_article_id"] = trn_df["click_article_id"].astype(int)

    if offline:
        val_df = pd.read_csv(save_path / "val_user_item_feats_df_all.csv")
        val_df["click_article_id"] = val_df["click_article_id"].astype(int)
    else:
        val_df = None

    tst_df = pd.read_csv(save_path / "tst_user_item_feats_df_all.csv")
    if len(tst_df) > 0:
        tst_df["click_article_id"] = tst_df["click_article_id"].astype(int)
    if "label" in tst_df.columns:
        del tst_df["label"]

    click_hist_all = pd.read_csv(save_path / "click_hist_all.csv")
    return LoadedFeatureFrames(
        trn_user_item_feats_df=trn_df,
        val_user_item_feats_df=val_df,
        tst_user_item_feats_df=tst_df,
        click_hist_all=click_hist_all,
        offline=offline,
    )


def build_history_behavior(click_hist_all: pd.DataFrame) -> pd.DataFrame:
    hist_click = (
        click_hist_all[["user_id", "click_article_id"]]
        .groupby("user_id")
        .agg(list)
        .reset_index()
    )
    his_behavior_df = pd.DataFrame()
    his_behavior_df["user_id"] = hist_click["user_id"]
    his_behavior_df[HISTORY_FEATURE] = hist_click["click_article_id"]
    return his_behavior_df


def merge_history_frames(loaded_frames: LoadedFeatureFrames) -> MergedFeatureFrames:
    his_behavior_df = build_history_behavior(loaded_frames.click_hist_all)

    trn_df = loaded_frames.trn_user_item_feats_df.merge(his_behavior_df, on="user_id")
    if loaded_frames.offline and loaded_frames.val_user_item_feats_df is not None:
        val_df = loaded_frames.val_user_item_feats_df.merge(his_behavior_df, on="user_id")
    else:
        val_df = None

    if len(loaded_frames.tst_user_item_feats_df) > 0:
        tst_df = loaded_frames.tst_user_item_feats_df.merge(his_behavior_df, on="user_id")
    else:
        tst_df = loaded_frames.tst_user_item_feats_df.copy()

    if loaded_frames.offline and val_df is not None:
        val_hit_mask = val_df.groupby("user_id")["label"].transform("max") == 1
        val_df_hit = val_df[val_hit_mask].copy()
    else:
        val_hit_mask = None
        val_df_hit = None

    return MergedFeatureFrames(
        trn_df=trn_df,
        val_df=val_df,
        tst_df=tst_df,
        his_behavior_df=his_behavior_df,
        val_hit_mask=val_hit_mask,
        val_df_hit=val_df_hit,
        offline=loaded_frames.offline,
    )


def normalize_dense_features(
    trn_df: pd.DataFrame,
    val_df: Optional[pd.DataFrame],
    tst_df: pd.DataFrame,
    dense_features: Sequence[str] = DENSE_FEATURES,
) -> Dict[str, MinMaxScaler]:
    mm_scalers: Dict[str, MinMaxScaler] = {}
    for feat in dense_features:
        scaler = MinMaxScaler()
        trn_df[feat] = scaler.fit_transform(trn_df[[feat]])
        if val_df is not None:
            val_df[feat] = scaler.transform(val_df[[feat]])
        if len(tst_df) > 0:
            tst_df[feat] = scaler.transform(tst_df[[feat]])
        mm_scalers[feat] = scaler
    return mm_scalers


def build_sparse_maps(
    frames: Sequence[pd.DataFrame],
    sparse_features: Sequence[str] = SPARSE_FEATURES,
    history_feature: str = HISTORY_FEATURE,
) -> Dict[str, Dict[int, int]]:
    sparse_maps: Dict[str, Dict[int, int]] = {}
    for feat in sparse_features:
        vals = pd.concat([frame[feat].astype("int64") for frame in frames], ignore_index=True)
        if feat == "click_article_id":
            hist_vals: List[int] = []
            for frame in frames:
                for seq in frame[history_feature]:
                    if isinstance(seq, (list, np.ndarray)):
                        hist_vals.extend(int(x) for x in seq)
            if hist_vals:
                vals = pd.concat([vals, pd.Series(hist_vals, dtype="int64")], ignore_index=True)
        _, uniques = pd.factorize(vals, sort=False)
        mapping = {int(val): int(idx) for idx, val in enumerate(uniques)}
        if feat == "click_article_id":
            mapping = {key: value + 1 for key, value in mapping.items()}
        sparse_maps[feat] = mapping
    return sparse_maps


def _vocab_size(mapping: Dict[int, int]) -> int:
    return max(mapping.values()) + 1 if mapping else 1


def pad_sequences_np(sequences: Sequence[Sequence[int]], maxlen: int, value: int = 0) -> np.ndarray:
    padded = np.full((len(sequences), maxlen), value, dtype=np.int64)
    for idx, seq in enumerate(sequences):
        if len(seq) == 0:
            continue
        trunc = seq[:maxlen]
        padded[idx, : len(trunc)] = np.asarray(trunc, dtype=np.int64)
    return padded


def build_model_input(
    df: pd.DataFrame,
    sparse_maps: Dict[str, Dict[int, int]],
    sparse_features: Sequence[str] = SPARSE_FEATURES,
    dense_features: Sequence[str] = DENSE_FEATURES,
    max_len: int = DEFAULT_MAX_HIST_LEN,
) -> Dict[str, np.ndarray]:
    model_input: Dict[str, np.ndarray] = {}
    for feat in sparse_features:
        model_input[feat] = (
            df[feat].astype("int64").map(sparse_maps[feat]).fillna(0).astype("int64").values
        )
    for feat in dense_features:
        model_input[feat] = df[feat].astype("float32").values

    click_mapping = sparse_maps["click_article_id"]
    raw_seq_list = df[HISTORY_FEATURE].tolist()
    mapped_seq: List[List[int]] = []
    for seq in raw_seq_list:
        if isinstance(seq, (list, np.ndarray)):
            mapped_seq.append([int(click_mapping.get(int(x), 0)) if int(x) != 0 else 0 for x in seq])
        else:
            mapped_seq.append([0])
    model_input[HISTORY_FEATURE] = pad_sequences_np(mapped_seq, maxlen=max_len, value=0)
    return model_input


def prepare_model_inputs(
    merged_frames: MergedFeatureFrames,
    emb_dim: int = DEFAULT_EMB_DIM,
    max_len: int = DEFAULT_MAX_HIST_LEN,
) -> PreparedRankingData:
    trn_df = merged_frames.trn_df.copy()
    val_df = merged_frames.val_df.copy() if merged_frames.val_df is not None else None
    tst_df = merged_frames.tst_df.copy()

    mm_scalers = normalize_dense_features(trn_df, val_df, tst_df)
    all_frames = [trn_df] + ([val_df] if val_df is not None else []) + ([tst_df] if len(tst_df) > 0 else [])
    sparse_maps = build_sparse_maps(all_frames)
    sparse_vocab_sizes = {feat: _vocab_size(sparse_maps[feat]) for feat in SPARSE_FEATURES}

    x_trn = build_model_input(trn_df, sparse_maps, max_len=max_len)
    y_trn = trn_df["label"].values.astype("float32")

    if val_df is not None:
        x_val = build_model_input(val_df, sparse_maps, max_len=max_len)
        y_val = val_df["label"].values.astype("float32")
    else:
        x_val, y_val = None, None

    if len(tst_df) > 0:
        x_tst = build_model_input(tst_df, sparse_maps, max_len=max_len)
    else:
        x_tst = None

    return PreparedRankingData(
        trn_df=trn_df,
        val_df=val_df,
        tst_df=tst_df,
        val_hit_mask=merged_frames.val_hit_mask,
        val_df_hit=merged_frames.val_df_hit,
        sparse_maps=sparse_maps,
        sparse_vocab_sizes=sparse_vocab_sizes,
        mm_scalers=mm_scalers,
        x_trn=x_trn,
        y_trn=y_trn,
        x_val=x_val,
        y_val=y_val,
        x_tst=x_tst,
        dense_input_dim=len(DENSE_FEATURES),
        emb_dim=emb_dim,
        max_len=max_len,
    )


def make_mlp(
    input_dim: int,
    hidden_units: Sequence[int],
    dropout_rate: float = 0.0,
    output_dim: Optional[int] = None,
    output_activation: Optional[str] = None,
):
    ensure_torch_available()
    layers: List[nn.Module] = []
    prev_dim = input_dim
    for units in hidden_units:
        layers.append(nn.Linear(prev_dim, units))
        layers.append(nn.ReLU())
        if dropout_rate > 0:
            layers.append(nn.Dropout(dropout_rate))
        prev_dim = units
    if output_dim is not None:
        layers.append(nn.Linear(prev_dim, output_dim))
        prev_dim = output_dim
        if output_activation == "sigmoid":
            layers.append(nn.Sigmoid())
    return nn.Sequential(*layers), prev_dim


class DINAttentionPooling(TorchModuleBase):
    def __init__(self, emb_dim: int, hidden_units: Sequence[int]):
        super().__init__()
        self.att_mlp, _ = make_mlp(
            emb_dim * 4,
            hidden_units,
            dropout_rate=0.0,
            output_dim=1,
            output_activation=None,
        )

    def forward(self, query_emb: torch.Tensor, hist_emb: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
        query = query_emb.unsqueeze(1).expand_as(hist_emb)
        att_input = torch.cat([query, hist_emb, query - hist_emb, query * hist_emb], dim=-1)
        scores = self.att_mlp(att_input).squeeze(-1)

        all_pad = ~mask.any(dim=1)
        scores = scores.masked_fill(~mask, -1e9)
        scores = scores.masked_fill(all_pad.unsqueeze(1), 0.0)

        weights = torch.softmax(scores, dim=1)
        weights = weights * mask.float()
        denom = weights.sum(dim=1, keepdim=True).clamp_min(1e-8)
        weights = weights / denom

        pooled = torch.bmm(weights.unsqueeze(1), hist_emb).squeeze(1)
        pooled = pooled.masked_fill(all_pad.unsqueeze(1), 0.0)
        return pooled


class JRCBackboneInputEncoder(TorchModuleBase):
    def __init__(self, sparse_vocab_sizes: Dict[str, int], model_config: Dict[str, Any]):
        super().__init__()
        self.sparse_features = list(sparse_vocab_sizes.keys())
        self.dense_features = list(DENSE_FEATURES)
        self.emb_dim = model_config["emb_dim"]
        self.sparse_embeddings = nn.ModuleDict()
        for feat, vocab_size in sparse_vocab_sizes.items():
            if feat == "click_article_id":
                self.sparse_embeddings[feat] = nn.Embedding(vocab_size, self.emb_dim, padding_idx=0)
            else:
                self.sparse_embeddings[feat] = nn.Embedding(vocab_size, self.emb_dim)
        self.din_pooling = DINAttentionPooling(
            self.emb_dim,
            model_config.get("attention_hidden_units", [64, 32]),
        )

    def forward(self, batch: Dict[str, torch.Tensor]) -> Dict[str, Any]:
        sparse_emb_dict = {
            feat: self.sparse_embeddings[feat](batch[feat]) for feat in self.sparse_features
        }
        dense_tensor = torch.stack([batch[feat] for feat in self.dense_features], dim=1)

        target_emb = sparse_emb_dict["click_article_id"]
        hist_ids = batch[HISTORY_FEATURE]
        hist_emb = self.sparse_embeddings["click_article_id"](hist_ids)
        hist_mask = hist_ids.ne(0)
        hist_repr = self.din_pooling(target_emb, hist_emb, hist_mask)

        flat_sparse = [sparse_emb_dict[feat] for feat in self.sparse_features]
        flat_input = torch.cat(flat_sparse + [dense_tensor, hist_repr], dim=1)
        return {
            "sparse_embeddings": sparse_emb_dict,
            "dense_tensor": dense_tensor,
            "hist_repr": hist_repr,
            "flat_input": flat_input,
        }


class GroupSemanticTokenization(TorchModuleBase):
    def __init__(self, emb_dim: int, token_dim: int, verbose: bool = False):
        super().__init__()
        self.emb_dim = emb_dim
        self.token_dim = token_dim
        self.group_order = list(TOKEN_GROUPS.keys())
        self.group_input_dims: Dict[str, int] = {}
        self.projections = nn.ModuleDict()
        self._validate_group_coverage()

        for group_name, spec in TOKEN_GROUPS.items():
            if spec["kind"] == "sparse":
                input_dim = len(spec["features"]) * emb_dim
            elif spec["kind"] == "dense":
                input_dim = len(spec["features"])
            elif spec["kind"] == "history":
                input_dim = emb_dim
            else:  # pragma: no cover - defensive
                raise ValueError(f"Unknown token group kind: {spec['kind']}")

            self.group_input_dims[group_name] = input_dim
            self.projections[group_name] = nn.Linear(input_dim, token_dim)

        if verbose:
            self.print_summary()

    def _validate_group_coverage(self):
        sparse_consumed: List[str] = []
        dense_consumed: List[str] = []
        for spec in TOKEN_GROUPS.values():
            if spec["kind"] == "sparse":
                sparse_consumed.extend(spec["features"])
            elif spec["kind"] == "dense":
                dense_consumed.extend(spec["features"])

        assert len(sparse_consumed) == len(set(sparse_consumed)), "Sparse token groups contain duplicates."
        assert len(dense_consumed) == len(set(dense_consumed)), "Dense token groups contain duplicates."
        assert set(sparse_consumed) == set(SPARSE_FEATURES), "Sparse features are not consumed exactly once."
        assert set(dense_consumed) == set(DENSE_FEATURES), "Dense features are not consumed exactly once."

    def print_summary(self):
        print("GroupSemanticTokenization summary:")
        for group_name in self.group_order:
            print(f"  {group_name}: input_dim={self.group_input_dims[group_name]}")
        print(f"  num_tokens={len(self.group_order)}")

    def forward(
        self,
        batch: Dict[str, torch.Tensor],
        sparse_embeddings: Dict[str, torch.Tensor],
        hist_repr: torch.Tensor,
    ) -> torch.Tensor:
        tokens: List[torch.Tensor] = []
        for group_name, spec in TOKEN_GROUPS.items():
            if spec["kind"] == "sparse":
                group_input = torch.cat([sparse_embeddings[feat] for feat in spec["features"]], dim=1)
            elif spec["kind"] == "dense":
                group_input = torch.stack([batch[feat] for feat in spec["features"]], dim=1)
            else:
                group_input = hist_repr
            tokens.append(self.projections[group_name](group_input))
        return torch.stack(tokens, dim=1)


class TokenMixer(TorchModuleBase):
    def __init__(self, num_T: int, token_dim: int, num_H: int):
        super().__init__()
        assert num_H == num_T, "RankMixer requires num_H == num_T for shape-preserving token mixing."
        assert token_dim % num_H == 0, "token_dim must be divisible by num_H."
        self.num_T = num_T
        self.num_H = num_H
        self.d_k = token_dim // num_H

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        batch_size = x.size(0)
        x = x.reshape(batch_size, self.num_T, self.num_H, self.d_k)
        x = x.permute(0, 2, 1, 3).contiguous()
        return x.reshape(batch_size, self.num_H, self.num_T * self.d_k)


class PerTokenFFN(TorchModuleBase):
    def __init__(self, num_T: int, token_dim: int, expansion_ratio: int = 4):
        super().__init__()
        self.experts = nn.ModuleList(
            [
                nn.Sequential(
                    nn.Linear(token_dim, token_dim * expansion_ratio),
                    nn.GELU(),
                    nn.Linear(token_dim * expansion_ratio, token_dim),
                )
                for _ in range(num_T)
            ]
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return torch.stack(
            [self.experts[idx](x[:, idx]) for idx in range(len(self.experts))],
            dim=1,
        )


class RankMixerLayer(TorchModuleBase):
    def __init__(self, num_T: int, token_dim: int, num_H: int, expansion_ratio: int = 4):
        super().__init__()
        self.token_mixer = TokenMixer(num_T, token_dim, num_H)
        self.per_token_ffn = PerTokenFFN(num_T, token_dim, expansion_ratio)
        self.norm1 = nn.LayerNorm(token_dim)
        self.norm2 = nn.LayerNorm(token_dim)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        mixed_x = self.token_mixer(x)
        x = self.norm1(x + mixed_x)
        x = self.norm2(x + self.per_token_ffn(x))
        return x


class RankMixerBackbone(TorchModuleBase):
    def __init__(
        self,
        emb_dim: int,
        token_dim: int,
        num_layers: int = 2,
        num_T: int = 6,
        num_H: int = 6,
        expansion_ratio: int = 4,
        verbose: bool = False,
    ):
        super().__init__()
        assert num_T == len(TOKEN_GROUPS), "num_T must match the fixed number of semantic token groups."
        assert num_H == num_T, "num_H must equal num_T."
        assert token_dim % num_H == 0, "token_dim must be divisible by num_H."
        self.num_T = num_T
        self.token_dim = token_dim
        self.tokenization = GroupSemanticTokenization(emb_dim, token_dim, verbose=verbose)
        self.layers = nn.ModuleList(
            [RankMixerLayer(num_T, token_dim, num_H, expansion_ratio) for _ in range(num_layers)]
        )

    def forward(
        self,
        batch: Dict[str, torch.Tensor],
        sparse_embeddings: Dict[str, torch.Tensor],
        hist_repr: torch.Tensor,
    ) -> torch.Tensor:
        tokens = self.tokenization(batch, sparse_embeddings, hist_repr)
        for layer in self.layers:
            tokens = layer(tokens)
        return tokens.mean(dim=1)


class JRCWithMLPBackbone(TorchModuleBase):
    def __init__(self, sparse_vocab_sizes: Dict[str, int], dense_input_dim: int, model_config: Dict[str, Any]):
        super().__init__()
        self.input_encoder = JRCBackboneInputEncoder(sparse_vocab_sizes, model_config)
        shared_input_dim = len(sparse_vocab_sizes) * model_config["emb_dim"] + dense_input_dim + model_config["emb_dim"]
        self.shared_backbone, shared_output_dim = make_mlp(
            shared_input_dim,
            model_config.get("shared_dnn_units", [256, 128]),
            dropout_rate=model_config.get("dropout_rate", 0.0),
            output_dim=None,
        )
        self.logit_head, logit_hidden_dim = make_mlp(
            shared_output_dim,
            model_config.get("logit_head_units", [64]),
            dropout_rate=model_config.get("dropout_rate", 0.0),
            output_dim=None,
        )
        self.logits_out = nn.Linear(logit_hidden_dim, 2)

    def forward(self, batch: Dict[str, torch.Tensor]) -> torch.Tensor:
        encoded = self.input_encoder(batch)
        shared_repr = self.shared_backbone(encoded["flat_input"])
        logits_hidden = self.logit_head(shared_repr)
        return self.logits_out(logits_hidden)


class PyTorchJRCWithRankMixerBackbone(TorchModuleBase):
    def __init__(
        self,
        sparse_vocab_sizes: Dict[str, int],
        dense_input_dim: int,
        model_config: Dict[str, Any],
        rankmixer_config: Dict[str, Any],
        verbose: bool = False,
    ):
        super().__init__()
        self.input_encoder = JRCBackboneInputEncoder(sparse_vocab_sizes, model_config)
        self.shared_backbone = RankMixerBackbone(
            emb_dim=model_config["emb_dim"],
            token_dim=rankmixer_config["token_dim"],
            num_layers=rankmixer_config["num_layers"],
            num_T=rankmixer_config.get("num_T", len(TOKEN_GROUPS)),
            num_H=rankmixer_config.get("num_H", len(TOKEN_GROUPS)),
            expansion_ratio=rankmixer_config["expansion_ratio"],
            verbose=verbose,
        )
        self.logit_head, logit_hidden_dim = make_mlp(
            rankmixer_config["token_dim"],
            model_config.get("logit_head_units", [64]),
            dropout_rate=model_config.get("dropout_rate", 0.0),
            output_dim=None,
        )
        self.logits_out = nn.Linear(logit_hidden_dim, 2)

    def forward(self, batch: Dict[str, torch.Tensor]) -> torch.Tensor:
        encoded = self.input_encoder(batch)
        shared_repr = self.shared_backbone(
            batch=batch,
            sparse_embeddings=encoded["sparse_embeddings"],
            hist_repr=encoded["hist_repr"],
        )
        logits_hidden = self.logit_head(shared_repr)
        return self.logits_out(logits_hidden)


def count_parameters(model: nn.Module) -> int:
    return sum(param.numel() for param in model.parameters() if param.requires_grad)


def build_baseline_model(
    sparse_vocab_sizes: Dict[str, int],
    dense_input_dim: int,
    model_config: Dict[str, Any],
    device: Optional[torch.device] = None,
) -> nn.Module:
    ensure_torch_available()
    model = JRCWithMLPBackbone(sparse_vocab_sizes, dense_input_dim, model_config)
    if device is not None:
        model = model.to(device)
    return model


def build_rankmixer_model(
    sparse_vocab_sizes: Dict[str, int],
    dense_input_dim: int,
    model_config: Dict[str, Any],
    rankmixer_config: Dict[str, Any],
    device: Optional[torch.device] = None,
    verbose: bool = False,
) -> nn.Module:
    ensure_torch_available()
    model = PyTorchJRCWithRankMixerBackbone(
        sparse_vocab_sizes,
        dense_input_dim,
        model_config,
        rankmixer_config,
        verbose=verbose,
    )
    if device is not None:
        model = model.to(device)
    return model


def search_rankmixer_config(
    target_params: int,
    sparse_vocab_sizes: Dict[str, int],
    dense_input_dim: int,
    model_config: Dict[str, Any],
    search_space: Dict[str, Any] = DEFAULT_RANKMIXER_SEARCH_SPACE,
) -> Dict[str, Any]:
    candidates: List[Dict[str, Any]] = []
    for token_dim in search_space["token_dim"]:
        for num_layers in search_space["num_layers"]:
            for expansion_ratio in search_space["expansion_ratio"]:
                candidate_config = {
                    "token_dim": token_dim,
                    "num_layers": num_layers,
                    "expansion_ratio": expansion_ratio,
                    "num_T": search_space["num_T"],
                    "num_H": search_space["num_H"],
                }
                model = build_rankmixer_model(
                    sparse_vocab_sizes,
                    dense_input_dim,
                    model_config,
                    candidate_config,
                    device=None,
                    verbose=False,
                )
                params = count_parameters(model)
                diff = abs(params - target_params)
                candidates.append(
                    {
                        "config": candidate_config,
                        "total_trainable_params": params,
                        "param_diff": diff,
                        "params_ratio_vs_baseline": params / target_params,
                    }
                )

    candidates.sort(
        key=lambda item: (
            item["param_diff"],
            abs(item["params_ratio_vs_baseline"] - 1.0),
            item["config"]["num_layers"],
            item["config"]["token_dim"],
            item["config"]["expansion_ratio"],
        )
    )
    best = candidates[0]
    return {
        "config": best["config"],
        "rankmixer_params": best["total_trainable_params"],
        "params_ratio_vs_baseline": best["params_ratio_vs_baseline"],
        "candidates": candidates,
        "candidates_df": pd.DataFrame(
            [
                {
                    "token_dim": item["config"]["token_dim"],
                    "num_layers": item["config"]["num_layers"],
                    "expansion_ratio": item["config"]["expansion_ratio"],
                    "total_trainable_params": item["total_trainable_params"],
                    "param_diff": item["param_diff"],
                    "params_ratio_vs_baseline": item["params_ratio_vs_baseline"],
                }
                for item in candidates
            ]
        ),
    }


class RankingDictDataset(TorchDatasetBase):
    def __init__(self, x_dict: Dict[str, np.ndarray], y: Optional[np.ndarray] = None):
        self.x_dict = x_dict
        self.y = y
        self.length = len(next(iter(x_dict.values())))

    def __len__(self) -> int:
        return self.length

    def __getitem__(self, idx: int):
        sample = {key: value[idx] for key, value in self.x_dict.items()}
        if self.y is None:
            return sample
        return sample, self.y[idx]


class UserGroupedBatchSampler:
    def __init__(self, user_ids: Sequence[int], batch_size: int = 256, shuffle: bool = True, seed: Optional[int] = None):
        self.user_ids = np.asarray(user_ids)
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.seed = seed
        self.epoch = 0
        user_to_indices: Dict[int, List[int]] = defaultdict(list)
        for idx, uid in enumerate(self.user_ids):
            user_to_indices[int(uid)].append(idx)
        self.user_blocks = list(user_to_indices.values())

    def __iter__(self):
        blocks = list(self.user_blocks)
        if self.shuffle:
            if self.seed is None:
                random.shuffle(blocks)
            else:
                rng = random.Random(self.seed + self.epoch)
                rng.shuffle(blocks)
        self.epoch += 1

        batch: List[int] = []
        for block in blocks:
            if len(block) >= self.batch_size:
                if batch:
                    yield batch
                    batch = []
                yield block
                continue
            if len(batch) + len(block) > self.batch_size:
                if batch:
                    yield batch
                batch = list(block)
            else:
                batch.extend(block)

        if batch:
            yield batch

    def __len__(self) -> int:
        count = 0
        batch: List[int] = []
        for block in self.user_blocks:
            if len(block) >= self.batch_size:
                if batch:
                    count += 1
                    batch = []
                count += 1
                continue
            if len(batch) + len(block) > self.batch_size:
                count += 1
                batch = list(block)
            else:
                batch.extend(block)
        if batch:
            count += 1
        return count


def collate_with_labels(batch):
    features, labels = zip(*batch)
    collated = {}
    for key in features[0]:
        arr = np.stack([item[key] for item in features])
        dtype = torch.long if key in SPARSE_FEATURES or key == HISTORY_FEATURE else torch.float32
        collated[key] = torch.tensor(arr, dtype=dtype)
    return collated, torch.tensor(np.asarray(labels), dtype=torch.float32)


def collate_features(batch):
    collated = {}
    for key in batch[0]:
        arr = np.stack([item[key] for item in batch])
        dtype = torch.long if key in SPARSE_FEATURES or key == HISTORY_FEATURE else torch.float32
        collated[key] = torch.tensor(arr, dtype=dtype)
    return collated


def make_loader(
    x_dict: Dict[str, np.ndarray],
    y: Optional[np.ndarray] = None,
    batch_size: int = 256,
    shuffle: bool = False,
    group_by_user: bool = False,
    sampler_seed: Optional[int] = None,
) -> DataLoader:
    ensure_torch_available()
    dataset = RankingDictDataset(x_dict, y)
    if y is None:
        return DataLoader(dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_features)
    if group_by_user:
        batch_sampler = UserGroupedBatchSampler(
            x_dict["user_id"],
            batch_size=batch_size,
            shuffle=shuffle,
            seed=sampler_seed,
        )
        return DataLoader(dataset, batch_sampler=batch_sampler, collate_fn=collate_with_labels)
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, collate_fn=collate_with_labels)


def move_batch_to_device(batch: Dict[str, torch.Tensor], device: torch.device) -> Dict[str, torch.Tensor]:
    return {key: value.to(device) for key, value in batch.items()}


def safe_auc(labels: Sequence[float], preds: Sequence[float]) -> float:
    labels = np.asarray(labels)
    if np.unique(labels).size < 2:
        return float("nan")
    return float(roc_auc_score(labels, preds))


def compute_ndcg_at_k(labels: Sequence[float], preds: Sequence[float], user_id_list: Sequence[int], k: int = 5) -> float:
    group_score: Dict[int, List[float]] = defaultdict(list)
    group_truth: Dict[int, List[float]] = defaultdict(list)
    for idx, uid in enumerate(user_id_list):
        group_score[int(uid)].append(float(preds[idx]))
        group_truth[int(uid)].append(float(labels[idx]))
    ndcg_list: List[float] = []
    for uid in group_score:
        y_true = np.array([group_truth[uid]])
        y_score = np.array([group_score[uid]])
        if np.sum(y_true) == 0:
            ndcg_list.append(0.0)
        else:
            ndcg_list.append(float(ndcg_score(y_true, y_score, k=k)))
    return float(np.mean(ndcg_list))


def compute_mrr_hr_stats(df: pd.DataFrame, topk: int = 5, score_col: str = "pred_score") -> Dict[str, float]:
    df_sorted = df.sort_values(by=["user_id", score_col], ascending=[True, False])
    mrr_sum = 0.0
    hr_sum = 0.0
    user_count = 0
    pos_user_count = 0
    for _, group in df_sorted.groupby("user_id"):
        user_count += 1
        labels = group["label"].values
        if labels.sum() == 0:
            continue
        pos_user_count += 1
        pos_idx = np.where(labels == 1)[0]
        if len(pos_idx) > 0:
            rank = int(pos_idx[0]) + 1
            if rank <= topk:
                mrr_sum += 1.0 / rank
                hr_sum += 1.0
    return {
        "mrr5": float(mrr_sum / user_count if user_count > 0 else 0.0),
        "hr5": float(hr_sum / user_count if user_count > 0 else 0.0),
        "pos_user_count": int(pos_user_count),
        "user_count": int(user_count),
    }


def evaluate_prediction_slice(labels: Sequence[float], preds: Sequence[float], eval_df: pd.DataFrame, topk: int = 5) -> Dict[str, float]:
    eval_frame = eval_df[["user_id", "click_article_id", "label"]].copy()
    eval_frame["pred_score"] = preds
    rank_metrics = compute_mrr_hr_stats(eval_frame, topk=topk)
    return {
        "auc": safe_auc(labels, preds),
        "ndcg5": compute_ndcg_at_k(labels, preds, eval_df["user_id"].values, k=topk),
        "mrr5": rank_metrics["mrr5"],
        "hr5": rank_metrics["hr5"],
        "pos_user_count": rank_metrics["pos_user_count"],
        "user_count": rank_metrics["user_count"],
    }


def build_experiment_pred_score(logits_2d: np.ndarray, score_mode: str = "diff") -> np.ndarray:
    if score_mode == "diff":
        return logits_2d[:, 1] - logits_2d[:, 0]
    if score_mode == "click":
        return logits_2d[:, 1]
    raise ValueError(f"Unknown score_mode: {score_mode}")


def evaluate_experiment_from_raw(
    logits_2d: np.ndarray,
    y_true: np.ndarray,
    val_df: pd.DataFrame,
    val_df_hit: Optional[pd.DataFrame] = None,
    val_hit_mask: Optional[pd.Series] = None,
    topk: int = 5,
    score_mode: str = "diff",
) -> Dict[str, Any]:
    preds = build_experiment_pred_score(logits_2d, score_mode=score_mode)
    metrics: Dict[str, Any] = {"preds": preds, "logits_2d": logits_2d}

    full_metrics = evaluate_prediction_slice(y_true, preds, val_df, topk=topk)
    for key, value in full_metrics.items():
        metrics[f"full_{key}"] = value

    if val_df_hit is not None and len(val_df_hit) > 0 and val_hit_mask is not None:
        hit_mask_np = val_hit_mask.to_numpy() if hasattr(val_hit_mask, "to_numpy") else np.asarray(val_hit_mask)
        hit_preds = preds[hit_mask_np]
        hit_labels = val_df_hit["label"].values
        hit_metrics = evaluate_prediction_slice(hit_labels, hit_preds, val_df_hit, topk=topk)
        metrics["hit_preds"] = hit_preds
        for key, value in hit_metrics.items():
            metrics[f"hit_{key}"] = value
    else:
        for key in ["auc", "ndcg5", "mrr5", "hr5"]:
            metrics[f"hit_{key}"] = float("nan")
        metrics["hit_pos_user_count"] = 0
        metrics["hit_user_count"] = 0
    return metrics


def paper_context_ge_loss_old(logits_2d: torch.Tensor, labels_onehot: torch.Tensor, context_ids: torch.Tensor) -> torch.Tensor:
    batch_size = logits_2d.size(0)
    if batch_size == 0:
        return logits_2d.new_tensor(0.0)

    mask = context_ids.unsqueeze(0).eq(context_ids.unsqueeze(1))
    mask_f = mask.float()
    logits_tile = logits_2d.unsqueeze(1).expand(-1, batch_size, -1).clone()
    labels_tile = labels_onehot.unsqueeze(1).expand(-1, batch_size, -1).float()
    invalid_mask = ~mask.unsqueeze(-1)
    logits_tile = logits_tile.masked_fill(invalid_mask, -1e9)
    labels_tile = labels_tile * mask.unsqueeze(-1).float()

    l_neg = logits_tile[:, :, 0]
    l_pos = logits_tile[:, :, 1]
    y_neg = labels_tile[:, :, 0]
    y_pos = labels_tile[:, :, 1]

    log_softmax_pos = F.log_softmax(l_pos, dim=0)
    log_softmax_neg = F.log_softmax(l_neg, dim=0)

    loss_pos = -(y_pos * log_softmax_pos).sum(dim=0)
    loss_neg = -(y_neg * log_softmax_neg).sum(dim=0)

    context_sizes = mask_f.sum(dim=0).clamp_min(1.0)
    return ((loss_pos + loss_neg) / context_sizes).mean()


def paper_context_ge_loss_balanced(
    logits_2d: torch.Tensor,
    labels_onehot: torch.Tensor,
    context_ids: torch.Tensor,
    pos_weight: float = 1.0,
    neg_weight: float = 1.0,
) -> torch.Tensor:
    batch_size = logits_2d.size(0)
    if batch_size == 0:
        return logits_2d.new_tensor(0.0)

    mask = context_ids.unsqueeze(0).eq(context_ids.unsqueeze(1))
    logits_tile = logits_2d.unsqueeze(1).expand(-1, batch_size, -1).clone()
    labels_tile = labels_onehot.unsqueeze(1).expand(-1, batch_size, -1).float()
    invalid_mask = ~mask.unsqueeze(-1)
    logits_tile = logits_tile.masked_fill(invalid_mask, -1e9)
    labels_tile = labels_tile * mask.unsqueeze(-1).float()

    l_neg = logits_tile[:, :, 0]
    l_pos = logits_tile[:, :, 1]
    y_neg = labels_tile[:, :, 0]
    y_pos = labels_tile[:, :, 1]

    log_softmax_pos = F.log_softmax(l_pos, dim=0)
    log_softmax_neg = F.log_softmax(l_neg, dim=0)

    loss_pos = -(y_pos * log_softmax_pos).sum(dim=0)
    loss_neg = -(y_neg * log_softmax_neg).sum(dim=0)

    pos_cnt = y_pos.sum(dim=0).clamp_min(1.0)
    neg_cnt = y_neg.sum(dim=0).clamp_min(1.0)
    return (pos_weight * loss_pos / pos_cnt + neg_weight * loss_neg / neg_cnt).mean()


def compute_joint_loss_ge_old(
    logits_2d: torch.Tensor,
    y_true: torch.Tensor,
    context_ids: torch.Tensor,
    alpha: float = 0.5,
):
    ce_loss = F.cross_entropy(logits_2d, y_true.long())
    y_true_long = y_true.long()
    y_neg = 1 - y_true_long
    labels_onehot = torch.stack([y_neg, y_true_long], dim=1).float()
    ge_loss = paper_context_ge_loss_old(logits_2d, labels_onehot, context_ids)
    total_loss = alpha * ce_loss + (1.0 - alpha) * ge_loss
    return total_loss, ce_loss, ge_loss


def compute_joint_loss_ge_balanced(
    logits_2d: torch.Tensor,
    y_true: torch.Tensor,
    context_ids: torch.Tensor,
    alpha: float = 0.5,
    pos_weight: float = 1.0,
    neg_weight: float = 1.0,
):
    ce_loss = F.cross_entropy(logits_2d, y_true.long())
    y_true_long = y_true.long()
    y_neg = 1 - y_true_long
    labels_onehot = torch.stack([y_neg, y_true_long], dim=1).float()
    ge_loss = paper_context_ge_loss_balanced(
        logits_2d,
        labels_onehot,
        context_ids,
        pos_weight=pos_weight,
        neg_weight=neg_weight,
    )
    total_loss = alpha * ce_loss + (1.0 - alpha) * ge_loss
    return total_loss, ce_loss, ge_loss


def raw_predict_logits(
    model: nn.Module,
    x_data: Dict[str, np.ndarray],
    batch_size: int = 256,
    device: Optional[torch.device] = None,
) -> np.ndarray:
    ensure_torch_available()
    if device is None:
        device = get_default_device()
    loader = make_loader(x_data, y=None, batch_size=batch_size, shuffle=False)
    model.eval()
    logits_list: List[np.ndarray] = []
    with torch.no_grad():
        for batch in loader:
            batch = move_batch_to_device(batch, device)
            logits_2d = model(batch)
            logits_list.append(logits_2d.cpu().numpy())
    return np.concatenate(logits_list, axis=0)


def train_ge_model(
    model: nn.Module,
    x_train: Dict[str, np.ndarray],
    y_train: np.ndarray,
    x_val: Optional[Dict[str, np.ndarray]] = None,
    y_val: Optional[np.ndarray] = None,
    val_df: Optional[pd.DataFrame] = None,
    val_df_hit: Optional[pd.DataFrame] = None,
    val_hit_mask: Optional[pd.Series] = None,
    epochs: int = 5,
    batch_size: int = 256,
    lr: float = 1e-3,
    select_topk: int = 5,
    loss_type: str = "balanced_ge",
    score_mode: str = "click",
    alpha: float = 0.5,
    pos_weight: float = 1.0,
    neg_weight: float = 1.0,
    seed: Optional[int] = None,
    device: Optional[torch.device] = None,
) -> Dict[str, Any]:
    ensure_torch_available()
    if device is None:
        device = get_default_device()
    if seed is not None:
        set_global_seed(seed)

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    train_loader = make_loader(
        x_train,
        y_train,
        batch_size=batch_size,
        shuffle=True,
        group_by_user=True,
        sampler_seed=seed,
    )

    history: Dict[str, List[float]] = {
        "total_loss": [],
        "ce_loss": [],
        "ge_loss": [],
        "full_auc": [],
        "full_ndcg5": [],
        "full_mrr5": [],
        "full_hr5": [],
        "hit_auc": [],
        "hit_ndcg5": [],
        "hit_mrr5": [],
        "hit_hr5": [],
    }
    best_record = {
        "metric": float("-inf"),
        "epoch": None,
        "state_dict": None,
        "eval_metrics": None,
    }

    for epoch in range(epochs):
        epoch_losses = {key: [] for key in ["total_loss", "ce_loss", "ge_loss"]}
        t0 = time.time()
        model.train()

        for x_batch, y_batch in train_loader:
            x_batch = move_batch_to_device(x_batch, device)
            y_batch = y_batch.to(device)
            context_ids = x_batch["user_id"]

            optimizer.zero_grad()
            logits_2d = model(x_batch)
            if loss_type == "old_ge":
                total_loss, ce_loss, ge_loss = compute_joint_loss_ge_old(
                    logits_2d,
                    y_batch,
                    context_ids,
                    alpha=alpha,
                )
            elif loss_type == "balanced_ge":
                total_loss, ce_loss, ge_loss = compute_joint_loss_ge_balanced(
                    logits_2d,
                    y_batch,
                    context_ids,
                    alpha=alpha,
                    pos_weight=pos_weight,
                    neg_weight=neg_weight,
                )
            else:
                raise ValueError(f"Unknown loss_type: {loss_type}")

            total_loss.backward()
            optimizer.step()

            epoch_losses["total_loss"].append(float(total_loss.item()))
            epoch_losses["ce_loss"].append(float(ce_loss.item()))
            epoch_losses["ge_loss"].append(float(ge_loss.item()))

        elapsed = time.time() - t0
        history["total_loss"].append(float(np.mean(epoch_losses["total_loss"])))
        history["ce_loss"].append(float(np.mean(epoch_losses["ce_loss"])))
        history["ge_loss"].append(float(np.mean(epoch_losses["ge_loss"])))

        log_str = (
            f"Epoch {epoch + 1}/{epochs} ({elapsed:.1f}s)  "
            f"Total: {history['total_loss'][-1]:.4f}  "
            f"CE: {history['ce_loss'][-1]:.4f}  "
            f"GE: {history['ge_loss'][-1]:.4f}"
        )

        if x_val is not None and y_val is not None and val_df is not None:
            val_logits_2d = raw_predict_logits(model, x_val, batch_size=batch_size, device=device)
            eval_metrics = evaluate_experiment_from_raw(
                val_logits_2d,
                y_val,
                val_df,
                val_df_hit=val_df_hit,
                val_hit_mask=val_hit_mask,
                topk=select_topk,
                score_mode=score_mode,
            )
            for metric_name in [
                "full_auc",
                "full_ndcg5",
                "full_mrr5",
                "full_hr5",
                "hit_auc",
                "hit_ndcg5",
                "hit_mrr5",
                "hit_hr5",
            ]:
                history[metric_name].append(float(eval_metrics.get(metric_name, float("nan"))))

            current_metric = eval_metrics["hit_mrr5"]
            if np.isnan(current_metric):
                current_metric = eval_metrics["full_mrr5"]
            if current_metric > best_record["metric"]:
                best_record["metric"] = float(current_metric)
                best_record["epoch"] = epoch + 1
                best_record["state_dict"] = copy.deepcopy(model.state_dict())
                best_record["eval_metrics"] = {
                    key: value
                    for key, value in eval_metrics.items()
                    if key not in {"preds", "hit_preds", "logits_2d"}
                }
                log_str += "  | best*"

            log_str += (
                f"  | Full MRR@{select_topk}: {eval_metrics['full_mrr5']:.5f}"
                f"  | Hit MRR@{select_topk}: {eval_metrics['hit_mrr5']:.5f}"
                f"  | Full NDCG@{select_topk}: {eval_metrics['full_ndcg5']:.4f}"
                f"  | Hit NDCG@{select_topk}: {eval_metrics['hit_ndcg5']:.4f}"
            )

        print(log_str)

    if best_record["state_dict"] is not None:
        model.load_state_dict(best_record["state_dict"])

    return {
        "history": history,
        "best_record": {
            "metric": None if best_record["metric"] == float("-inf") else float(best_record["metric"]),
            "epoch": best_record["epoch"],
            "state_dict": best_record["state_dict"],
            "eval_metrics": best_record["eval_metrics"],
        },
    }


def make_run_record(
    model_name: str,
    backbone_type: str,
    loss_type: str,
    score_mode: str,
    total_trainable_params: int,
    baseline_params: int,
    model_config: Dict[str, Any],
    train_result: Dict[str, Any],
) -> Dict[str, Any]:
    best_record = train_result["best_record"]
    eval_metrics = best_record.get("eval_metrics", {}) or {}
    return {
        "model_name": model_name,
        "backbone_type": backbone_type,
        "loss_type": loss_type,
        "score_mode": score_mode,
        "total_trainable_params": int(total_trainable_params),
        "params_ratio_vs_baseline": float(total_trainable_params / baseline_params),
        "best_epoch": best_record.get("epoch"),
        "best_hit_mrr5": eval_metrics.get("hit_mrr5", float("nan")),
        "best_full_mrr5": eval_metrics.get("full_mrr5", float("nan")),
        "best_hit_ndcg5": eval_metrics.get("hit_ndcg5", float("nan")),
        "best_full_ndcg5": eval_metrics.get("full_ndcg5", float("nan")),
        "best_hit_hr5": eval_metrics.get("hit_hr5", float("nan")),
        "best_full_hr5": eval_metrics.get("full_hr5", float("nan")),
        "full_auc_at_best": eval_metrics.get("full_auc", float("nan")),
        "hit_auc_at_best": eval_metrics.get("hit_auc", float("nan")),
        "history": train_result["history"],
        "best_record": best_record,
        "model_config": copy.deepcopy(model_config),
    }


def _selection_metric(record: Dict[str, Any]) -> float:
    hit_mrr5 = record.get("best_hit_mrr5", float("nan"))
    full_mrr5 = record.get("best_full_mrr5", float("nan"))
    if hit_mrr5 is not None and not np.isnan(hit_mrr5):
        return float(hit_mrr5)
    if full_mrr5 is not None and not np.isnan(full_mrr5):
        return float(full_mrr5)
    return float("-inf")


def flatten_run_record(record: Dict[str, Any]) -> Dict[str, Any]:
    return {
        "model_name": record["model_name"],
        "backbone_type": record["backbone_type"],
        "loss_type": record["loss_type"],
        "score_mode": record["score_mode"],
        "total_trainable_params": record["total_trainable_params"],
        "params_ratio_vs_baseline": record["params_ratio_vs_baseline"],
        "best_epoch": record["best_epoch"],
        "best_hit_mrr5": record["best_hit_mrr5"],
        "best_full_mrr5": record["best_full_mrr5"],
        "best_hit_ndcg5": record["best_hit_ndcg5"],
        "best_full_ndcg5": record["best_full_ndcg5"],
        "best_hit_hr5": record["best_hit_hr5"],
        "best_full_hr5": record["best_full_hr5"],
        "full_auc_at_best": record["full_auc_at_best"],
        "hit_auc_at_best": record["hit_auc_at_best"],
    }


def make_comparison_dataframe(records: Sequence[Dict[str, Any]]) -> pd.DataFrame:
    rows = [flatten_run_record(record) for record in records]
    compare_df = pd.DataFrame(rows)
    if len(compare_df) == 0:
        return compare_df
    compare_df["selection_metric"] = [
        _selection_metric(record) for record in records
    ]
    compare_df = compare_df.sort_values(
        ["selection_metric", "best_full_mrr5"],
        ascending=False,
    ).reset_index(drop=True)
    return compare_df.drop(columns=["selection_metric"])


def select_best_record(
    records: Sequence[Dict[str, Any]],
    backbone_type: Optional[str] = None,
) -> Optional[Dict[str, Any]]:
    filtered = [
        record for record in records if backbone_type is None or record["backbone_type"] == backbone_type
    ]
    if not filtered:
        return None
    return max(filtered, key=_selection_metric)


from IPython.display import display

ensure_torch_available()
set_global_seed(SEED)
DEVICE = get_default_device()

print(f"Device: {DEVICE}")
print(f"Sparse features: {len(SPARSE_FEATURES)}")
print(f"Dense features: {len(DENSE_FEATURES)}")
print(f"History feature: {HISTORY_FEATURE}")


Device: cpu
Sparse features: 9
Dense features: 20
History feature: hist_click_article_id


In [3]:
loaded_frames = load_feature_frames(SAVE_PATH, offline=offline)

print(f"Train feature frame: {loaded_frames.trn_user_item_feats_df.shape}")
if loaded_frames.val_user_item_feats_df is not None:
    print(f"Val feature frame: {loaded_frames.val_user_item_feats_df.shape}")
else:
    print("Val feature frame: None")
print(f"Test feature frame: {loaded_frames.tst_user_item_feats_df.shape}")
print(f"Click history frame: {loaded_frames.click_hist_all.shape}")


Train feature frame: (1426880, 31)
Val feature frame: (8000000, 31)
Test feature frame: (0, 31)
Click history frame: (912623, 9)


In [4]:
merged_frames = merge_history_frames(loaded_frames)

print(f"Merged train frame: {merged_frames.trn_df.shape}")
if merged_frames.val_df is not None:
    print(f"Merged val frame: {merged_frames.val_df.shape}")
else:
    print("Merged val frame: None")
print(f"Merged test frame: {merged_frames.tst_df.shape}")
print(f"History users: {merged_frames.his_behavior_df.shape}")
if merged_frames.val_df_hit is not None:
    print(f"Val hit users: {merged_frames.val_df_hit['user_id'].nunique()}")
    print(f"Val hit samples: {len(merged_frames.val_df_hit)}")


Merged train frame: (1426880, 32)
Merged val frame: (8000000, 32)
Merged test frame: (0, 31)
History users: (200000, 2)
Val hit users: 27382
Val hit samples: 5476400


In [5]:
prepared_data = prepare_model_inputs(
    merged_frames,
    emb_dim=DEFAULT_EMB_DIM,
    max_len=DEFAULT_MAX_HIST_LEN,
)

common_model_config = dict(DEFAULT_MODEL_CONFIG)
protocols = [
    {"loss_type": "balanced_ge", "score_mode": "click"},
    {"loss_type": "old_ge", "score_mode": "click"},
    {"loss_type": "old_ge", "score_mode": "diff"},
]

print(f"Train samples: {len(prepared_data.y_trn)}")
if prepared_data.y_val is not None:
    print(f"Val samples: {len(prepared_data.y_val)}")
print(f"Dense input dim: {prepared_data.dense_input_dim}")
print(f"Embedding dim: {prepared_data.emb_dim}")
print(f"Max history len: {prepared_data.max_len}")
print(f"Protocols: {protocols}")


Train samples: 1426880
Val samples: 8000000
Dense input dim: 20
Embedding dim: 8
Max history len: 50
Protocols: [{'loss_type': 'balanced_ge', 'score_mode': 'click'}, {'loss_type': 'old_ge', 'score_mode': 'click'}, {'loss_type': 'old_ge', 'score_mode': 'diff'}]


In [6]:
set_global_seed(SEED)
baseline_model = build_baseline_model(
    prepared_data.sparse_vocab_sizes,
    prepared_data.dense_input_dim,
    common_model_config,
    device=DEVICE,
)
baseline_params = count_parameters(baseline_model)
print(f"Baseline trainable params: {baseline_params:,}")
del baseline_model
clear_torch_cache()


Baseline trainable params: 1,454,179


In [7]:
search_result = search_rankmixer_config(
    target_params=baseline_params,
    sparse_vocab_sizes=prepared_data.sparse_vocab_sizes,
    dense_input_dim=prepared_data.dense_input_dim,
    model_config=common_model_config,
)
selected_rankmixer_config = search_result["config"]
print(f"Selected RankMixer config: {selected_rankmixer_config}")
print(f"Selected RankMixer params: {search_result['rankmixer_params']:,}")
print(f"Params ratio vs baseline: {search_result['params_ratio_vs_baseline']:.4f}")

rankmixer_preview_model = build_rankmixer_model(
    prepared_data.sparse_vocab_sizes,
    prepared_data.dense_input_dim,
    common_model_config,
    selected_rankmixer_config,
    device=DEVICE,
    verbose=True,
)
selected_rankmixer_params = count_parameters(rankmixer_preview_model)
print(f"Preview RankMixer params: {selected_rankmixer_params:,}")
del rankmixer_preview_model
clear_torch_cache()


Selected RankMixer config: {'token_dim': 48, 'num_layers': 1, 'expansion_ratio': 2, 'num_T': 6, 'num_H': 6}
Selected RankMixer params: 1,451,747
Params ratio vs baseline: 0.9983
GroupSemanticTokenization summary:
  user_context_sparse: input_dim=56
  item_sparse: input_dim=16
  recall_similarity_dense: input_dim=7
  ranking_meta_dense: input_dim=5
  user_article_profile_dense: input_dim=8
  history_token: input_dim=8
  num_tokens=6
Preview RankMixer params: 1,451,747


In [ ]:
all_run_records = []

for protocol in protocols:
    loss_type = protocol["loss_type"]
    score_mode = protocol["score_mode"]
    print("\n" + "=" * 70)
    print(f"Protocol: {loss_type} + {score_mode}")
    print("=" * 70)

    set_global_seed(SEED)
    mlp_model = build_baseline_model(
        prepared_data.sparse_vocab_sizes,
        prepared_data.dense_input_dim,
        common_model_config,
        device=DEVICE,
    )
    mlp_result = train_ge_model(
        mlp_model,
        prepared_data.x_trn,
        prepared_data.y_trn,
        x_val=prepared_data.x_val,
        y_val=prepared_data.y_val,
        val_df=prepared_data.val_df,
        val_df_hit=prepared_data.val_df_hit,
        val_hit_mask=prepared_data.val_hit_mask,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        lr=LR,
        select_topk=TOPK,
        loss_type=loss_type,
        score_mode=score_mode,
        alpha=ALPHA,
        seed=SEED,
        device=DEVICE,
    )
    all_run_records.append(
        make_run_record(
            model_name=f"mlp_{loss_type}_{score_mode}",
            backbone_type="mlp",
            loss_type=loss_type,
            score_mode=score_mode,
            total_trainable_params=baseline_params,
            baseline_params=baseline_params,
            model_config={"common": common_model_config},
            train_result=mlp_result,
        )
    )
    del mlp_model
    clear_torch_cache()

    set_global_seed(SEED)
    rankmixer_model = build_rankmixer_model(
        prepared_data.sparse_vocab_sizes,
        prepared_data.dense_input_dim,
        common_model_config,
        selected_rankmixer_config,
        device=DEVICE,
        verbose=False,
    )
    rankmixer_result = train_ge_model(
        rankmixer_model,
        prepared_data.x_trn,
        prepared_data.y_trn,
        x_val=prepared_data.x_val,
        y_val=prepared_data.y_val,
        val_df=prepared_data.val_df,
        val_df_hit=prepared_data.val_df_hit,
        val_hit_mask=prepared_data.val_hit_mask,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        lr=LR,
        select_topk=TOPK,
        loss_type=loss_type,
        score_mode=score_mode,
        alpha=ALPHA,
        seed=SEED,
        device=DEVICE,
    )
    all_run_records.append(
        make_run_record(
            model_name=f"rankmixer_{loss_type}_{score_mode}",
            backbone_type="rankmixer",
            loss_type=loss_type,
            score_mode=score_mode,
            total_trainable_params=selected_rankmixer_params,
            baseline_params=baseline_params,
            model_config={
                "common": common_model_config,
                "rankmixer": selected_rankmixer_config,
            },
            train_result=rankmixer_result,
        )
    )
    del rankmixer_model
    clear_torch_cache()

print(f"Completed runs: {len(all_run_records)}")



Protocol: balanced_ge + click
Epoch 1/5 (116.6s)  Total: 2.2815  CE: 0.2143  GE: 4.3486  | best*  | Full MRR@5: 0.17045  | Hit MRR@5: 0.24900  | Full NDCG@5: 0.2036  | Hit NDCG@5: 0.2974
Epoch 2/5 (88.9s)  Total: 2.1695  CE: 0.1979  GE: 4.1411  | best*  | Full MRR@5: 0.18520  | Hit MRR@5: 0.27054  | Full NDCG@5: 0.2190  | Hit NDCG@5: 0.3199
Epoch 3/5 (88.4s)  Total: 2.0986  CE: 0.1878  GE: 4.0094  | best*  | Full MRR@5: 0.19001  | Hit MRR@5: 0.27757  | Full NDCG@5: 0.2246  | Hit NDCG@5: 0.3281
Epoch 4/5 (87.1s)  Total: 2.0543  CE: 0.1812  GE: 3.9273  | Full MRR@5: 0.18889  | Hit MRR@5: 0.27593  | Full NDCG@5: 0.2232  | Hit NDCG@5: 0.3260
Epoch 5/5 (96.8s)  Total: 2.0235  CE: 0.1766  GE: 3.8704  | best*  | Full MRR@5: 0.19516  | Hit MRR@5: 0.28509  | Full NDCG@5: 0.2304  | Hit NDCG@5: 0.3366
Epoch 1/5 (108.5s)  Total: 2.2159  CE: 0.2059  GE: 4.2259  | best*  | Full MRR@5: 0.17745  | Hit MRR@5: 0.25921  | Full NDCG@5: 0.2124  | Hit NDCG@5: 0.3102
Epoch 2/5 (114.4s)  Total: 2.1000  CE: 0

In [ ]:
fair_ablation_compare_df = make_comparison_dataframe(all_run_records)
display(fair_ablation_compare_df)


In [ ]:
best_rankmixer_record = select_best_record(all_run_records, backbone_type="rankmixer")
best_rankmixer_row = flatten_run_record(best_rankmixer_record)
display(pd.DataFrame([best_rankmixer_row]))
print(f"Best RankMixer protocol: {best_rankmixer_record['loss_type']} + {best_rankmixer_record['score_mode']}")
print(f"Best RankMixer params ratio: {best_rankmixer_record['params_ratio_vs_baseline']:.4f}")
print(f"Best RankMixer config: {best_rankmixer_record['model_config']}")
